# Donut Form Extractor

This notebook uses Donut (Document understanding transformer) to extract information from contract forms identified in the preprocessing pipeline.
Only processes files where contains_form == T and analyzes the lowest page number from form_pages column.

In [85]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import json
import re
from typing import List, Dict, Any, Optional

import torch
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image
import fitz  # PyMuPDF
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [86]:
# Configuration
DATA_DIR = Path("../../data/raw/_contracts")
CSV_PATH = Path("../preprocessing/zero_shot_results_full_corpus.csv")  # Main results file
RESULTS_DIR = Path("../../data/intermediate_products/eds_forms_DONUT")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Processing controls
CLOBBER = True  # Set to False to skip files that already have processed output
TEST_MODE = True  # Set to True to run on limited number of files
TEST_LIMIT = 10  # Number of files to process in test mode

# Model configuration - Donut model
MODEL_NAME = "naver-clova-ix/donut-base-finetuned-cord-v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file: {CSV_PATH}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Clobber mode: {CLOBBER}")
print(f"Test mode: {TEST_MODE} (limit: {TEST_LIMIT if TEST_MODE else 'N/A'})")

Using device: cpu
Data directory: ../../data/raw/_contracts
CSV file: ../preprocessing/zero_shot_results_full_corpus.csv
Results directory: ../../data/intermediate_products/eds_forms_DONUT
Clobber mode: True
Test mode: True (limit: 10)


In [87]:
# Load and filter the CSV data
def load_and_filter_data(csv_path: Path) -> pd.DataFrame:
    """
    Load CSV and filter for files where contains_form == True
    """
    df = pd.read_csv(csv_path)
    print(f"Total files in CSV: {len(df)}")
    
    # Filter for files with forms
    form_files = df[df['contains_form'] == True].copy()
    print(f"Files with forms: {len(form_files)}")
    
    if len(form_files) == 0:
        print("Warning: No files found with contains_form == True")
        return pd.DataFrame()
    
    return form_files

# Load the data
df_forms = load_and_filter_data(CSV_PATH)

# Apply test mode limit if enabled
if TEST_MODE and not df_forms.empty:
    original_count = len(df_forms)
    df_forms = df_forms.head(TEST_LIMIT)
    print(f"\nTest mode enabled: Processing {len(df_forms)} files (out of {original_count} total)")

print("\nFirst few rows:")
print(df_forms.head())

Total files in CSV: 42490
Files with forms: 26109

Test mode enabled: Processing 10 files (out of 26109 total)

First few rows:
                               filename  contains_form      form_pages  \
804                           0-001.pdf           True  1,2,4,10,15,16   
956   0000000000000000000021689-004.pdf           True               5   
1660  0000000000000000000025661-003.pdf           True               5   
2895  0000000000000000000029856-000.pdf           True               1   
3996  0000000000000000000031808-000.pdf           True               1   

      num_form_pages  total_pages  max_similarity error  
804                6           34        0.981384   NaN  
956                1            5        0.908322   NaN  
1660               1            5        0.910548   NaN  
2895               1           10        0.963587   NaN  
3996               1           40        0.979984   NaN  


In [88]:
# Function to extract the lowest page number from form_pages column
def get_lowest_form_page(form_pages_str: str) -> Optional[int]:
    """
    Extract the lowest page number from the form_pages string
    Handles formats like: "1, 3, 5" or "[1, 3, 5]" or "1" or "[1]"
    """
    if pd.isna(form_pages_str) or form_pages_str == "":
        return None
    
    try:
        # Remove brackets and split by comma
        pages_str = str(form_pages_str).strip('[]')
        # Handle both comma-separated and single values
        if ',' in pages_str:
            pages = [int(p.strip()) for p in pages_str.split(',') if p.strip().isdigit()]
        else:
            pages = [int(pages_str.strip())] if pages_str.strip().isdigit() else []
        
        return min(pages) if pages else None
    except (ValueError, TypeError):
        print(f"Warning: Could not parse form_pages: {form_pages_str}")
        return None

def get_result_filename(filename: str) -> str:
    """
    Generate result filename: same as PDF filename but with .json extension
    """
    return Path(filename).with_suffix('.json').name

def result_exists(filename: str) -> bool:
    """
    Check if result file already exists for given input filename
    """
    result_file = RESULTS_DIR / get_result_filename(filename)
    return result_file.exists()

def filter_existing_results(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter out files that already have processed results
    """
    if df.empty:
        return df
    
    existing_mask = df['filename'].apply(result_exists)
    skipped_count = existing_mask.sum()
    
    if skipped_count > 0:
        print(f"Skipping {skipped_count} files with existing results")
    
    return df[~existing_mask].copy()

# Add column for target page number
if not df_forms.empty:
    df_forms['target_page'] = df_forms['form_pages'].apply(get_lowest_form_page)
    
    # Remove rows where we couldn't determine the target page
    df_forms = df_forms.dropna(subset=['target_page'])
    df_forms['target_page'] = df_forms['target_page'].astype(int)
    
    print(f"Files with valid target pages: {len(df_forms)}")
    print("\nTarget page distribution:")
    print(df_forms['target_page'].value_counts().sort_index())
    
    # Filter out files that already have results (if clobber is False)
    if not CLOBBER:
        df_forms = filter_existing_results(df_forms)
        print(f"\nAfter filtering existing results: {len(df_forms)} files to process")

Files with valid target pages: 10

Target page distribution:
1     4
3     1
4     2
5     2
14    1
Name: target_page, dtype: int64


In [89]:
# Initialize Donut model and processor
print("Loading Donut model and processor...")
processor = DonutProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

print("Model loaded successfully!")

Loading Donut model and processor...
Model loaded successfully!


In [90]:
# Define queries for information extraction
QUERIES = [
    "What is EDS Number?",
    "What is Date Prepared?"
]

print(f"Queries to process: {QUERIES}")

Queries to process: ['What is EDS Number?', 'What is Date Prepared?']


In [91]:
def pdf_page_to_image(pdf_path: Path, page_num: int, dpi: int = 150) -> Optional[Image.Image]:
    """
    Convert a specific PDF page to PIL Image
    """
    try:
        doc = fitz.open(str(pdf_path))
        if page_num < 1 or page_num > len(doc):
            print(f"Warning: Page {page_num} not found in {pdf_path.name} (total pages: {len(doc)})")
            doc.close()
            return None
        
        # Convert to 0-based index
        page = doc[page_num - 1]
        mat = fitz.Matrix(dpi/72, dpi/72)
        pix = page.get_pixmap(matrix=mat)
        img_data = pix.tobytes("ppm")
        doc.close()
        
        from io import BytesIO
        return Image.open(BytesIO(img_data))
    
    except Exception as e:
        print(f"Error converting PDF page {page_num} from {pdf_path}: {e}")
        return None

In [92]:
def extract_information_from_document(image: Image.Image, queries: List[str]) -> Dict[str, Any]:
    """
    Extract information from document image using Donut model with general document understanding
    """
    try:
        # Try different task prompts for better generalization
        task_prompts = [
            "<s_docvqa>",  # Document VQA (more general)
            "<s>",         # Generic prompt
            "<s_cord-v2>"  # Original CORD prompt (fallback)
        ]
        
        best_results = {}
        best_confidence = 0.0
        
        for task_prompt in task_prompts:
            try:
                # Prepare inputs
                pixel_values = processor(image, return_tensors="pt").pixel_values.to(DEVICE)
                
                # Create decoder inputs
                decoder_input_ids = processor.tokenizer(
                    task_prompt, 
                    add_special_tokens=False, 
                    return_tensors="pt"
                ).input_ids.to(DEVICE)
                
                # Generate output with updated cache handling
                with torch.no_grad():
                    outputs = model.generate(
                        pixel_values,
                        decoder_input_ids=decoder_input_ids,
                        max_length=model.decoder.config.max_position_embeddings,
                        early_stopping=True,
                        pad_token_id=processor.tokenizer.pad_token_id,
                        eos_token_id=processor.tokenizer.eos_token_id,
                        num_beams=1,
                        bad_words_ids=[[processor.tokenizer.unk_token_id]],
                        return_dict_in_generate=True,
                        # Remove use_cache to avoid deprecated past_key_values warning
                    )
                
                # Decode the output
                sequence = processor.batch_decode(outputs.sequences)[0]
                sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
                sequence = re.sub(r"<.*?>", "", sequence, count=1).strip()  # remove first task token
                
                print(f"Task prompt '{task_prompt}' output: {sequence[:200]}...")  # Debug output
                
                # Try to parse as JSON first
                try:
                    parsed_output = processor.token2json(sequence)
                    print(f"Successfully parsed JSON with '{task_prompt}': {parsed_output}")
                    
                    # Extract answers for our specific queries
                    results = {}
                    total_confidence = 0.0
                    
                    for query in queries:
                        query_key = query.lower().replace(' ', '_')
                        answer = ""
                        confidence = 0.0
                        
                        # Enhanced search for the query in the parsed output
                        if isinstance(parsed_output, dict):
                            # Multi-level search strategy
                            answer, confidence = search_for_query(parsed_output, query)
                        
                        results[f"{query_key}_answer"] = answer
                        results[f"{query_key}_confidence"] = confidence
                        total_confidence += confidence
                    
                    # If this prompt gives better overall results, use it
                    if total_confidence > best_confidence:
                        best_results = results
                        best_confidence = total_confidence
                        print(f"New best results with prompt '{task_prompt}' (total confidence: {total_confidence:.2f})")
                
                except Exception as json_error:
                    # Fallback: direct text extraction from sequence
                    print(f"JSON parsing failed for '{task_prompt}', trying direct extraction...")
                    results = extract_from_raw_text(sequence, queries)
                    
                    total_confidence = sum(results[f"{q.lower().replace(' ', '_')}_confidence"] for q in queries)
                    if total_confidence > best_confidence:
                        best_results = results
                        best_confidence = total_confidence
                        print(f"New best results from raw text with '{task_prompt}' (total confidence: {total_confidence:.2f})")
            
            except Exception as prompt_error:
                print(f"Error with task prompt '{task_prompt}': {prompt_error}")
                continue
        
        # Return best results found across all prompts
        if best_results:
            return best_results
        else:
            # No successful extraction with any prompt
            results = {}
            for query in queries:
                query_key = query.lower().replace(' ', '_')
                results[f"{query_key}_answer"] = ""
                results[f"{query_key}_confidence"] = 0.0
                results[f"{query_key}_error"] = "All task prompts failed"
            return results
    
    except Exception as e:
        print(f"Error during extraction: {e}")
        results = {}
        for query in queries:
            query_key = query.lower().replace(' ', '_')
            results[f"{query_key}_answer"] = ""
            results[f"{query_key}_confidence"] = 0.0
            results[f"{query_key}_error"] = str(e)
        return results

def search_for_query(parsed_output: dict, query: str) -> tuple:
    """
    Enhanced search function for finding query answers in parsed JSON
    """
    query_lower = query.lower()
    query_words = query_lower.split()
    
    # Strategy 1: Direct key matching
    for key, value in parsed_output.items():
        key_lower = str(key).lower()
        if (query_lower in key_lower or 
            all(word in key_lower for word in query_words) or
            # Specific patterns
            ('eds' in key_lower and 'eds' in query_lower) or
            ('number' in key_lower and 'number' in query_lower) or
            ('date' in key_lower and 'date' in query_lower) or
            ('prepared' in key_lower and 'prepared' in query_lower)):
            
            answer = str(value) if value else ""
            confidence = 0.9 if answer else 0.0
            print(f"Direct key match for '{query}': {key} -> {value}")
            return answer, confidence
    
    # Strategy 2: Value pattern matching
    for key, value in parsed_output.items():
        if isinstance(value, (str, int, float)):
            value_str = str(value)
            
            # EDS number patterns
            if 'eds' in query_lower:
                if (re.search(r'\b\d{4,}\b', value_str) or  # 4+ digit numbers
                    'eds' in value_str.lower()):
                    print(f"EDS pattern match: {value_str}")
                    return value_str, 0.7
            
            # Date patterns
            elif 'date' in query_lower:
                if re.search(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', value_str):
                    print(f"Date pattern match: {value_str}")
                    return value_str, 0.7
    
    # Strategy 3: Recursive nested search
    def recursive_search(obj, target_words):
        if isinstance(obj, dict):
            for k, v in obj.items():
                k_str = str(k).lower()
                if any(word in k_str for word in target_words):
                    return str(v) if v else "", 0.6
                result, conf = recursive_search(v, target_words)
                if result:
                    return result, conf * 0.8  # Slightly lower confidence for nested
        elif isinstance(obj, list):
            for item in obj:
                result, conf = recursive_search(item, target_words)
                if result:
                    return result, conf
        return "", 0.0
    
    nested_result, nested_conf = recursive_search(parsed_output, query_words)
    if nested_result:
        print(f"Nested search match for '{query}': {nested_result}")
        return nested_result, nested_conf
    
    return "", 0.0

def extract_from_raw_text(sequence: str, queries: List[str]) -> Dict[str, Any]:
    """
    Fallback extraction directly from raw model output using regex
    """
    results = {}
    
    for query in queries:
        query_key = query.lower().replace(' ', '_')
        answer = ""
        confidence = 0.0
        
        # Query-specific regex patterns
        if 'eds' in query.lower():
            patterns = [
                r'eds[\s#:]*([\\w\\d-]+)',
                r'(?:^|\s)([\d]{4,})(?=\s|$)',
                r'number[\s:]*([\\w\\d-]+)'
            ]
        elif 'date' in query.lower():
            patterns = [
                r'(?:date|prepared)[\s:]*([\d]{1,2}[/-][\d]{1,2}[/-][\d]{2,4})',
                r'([\w]+\s+[\d]{1,2},?\s+[\d]{4})',
                r'(?:^|\s)([\d]{1,2}[/-][\d]{1,2}[/-][\d]{2,4})(?=\s|$)'
            ]
        else:
            patterns = [rf'{re.escape(query)}[\s:]*([\\w\\d-]+)']
        
        # Try each pattern
        for pattern in patterns:
            match = re.search(pattern, sequence, re.IGNORECASE)
            if match:
                answer = match.group(1) if match.groups() else match.group(0)
                confidence = 0.5  # Lower confidence for fallback
                print(f"Regex fallback match for '{query}': {answer}")
                break
        
        results[f"{query_key}_answer"] = answer
        results[f"{query_key}_confidence"] = confidence
        if not answer:
            results[f"{query_key}_error"] = "No pattern match in raw text"
    
    return results

In [93]:
def save_individual_result(result: Dict[str, Any]):
    """
    Save individual result to separate JSON file
    """
    if not result['processed']:
        return
    
    result_file = RESULTS_DIR / get_result_filename(result['filename'])
    
    with open(result_file, 'w') as f:
        json.dump(result, f, indent=2)

def process_document(row: pd.Series) -> Dict[str, Any]:
    """
    Process a single document: convert PDF page to image and extract answers for all queries using Donut
    """
    filename = row['filename']
    target_page = int(row['target_page'])
    
    pdf_path = DATA_DIR / filename
    
    result = {
        'filename': filename,
        'target_page': target_page,
        'processed': False
    }
    
    # Check if file exists
    if not pdf_path.exists():
        result['error'] = f"File not found: {pdf_path}"
        return result
    
    # Convert PDF page to image
    image = pdf_page_to_image(pdf_path, target_page)
    if image is None:
        result['error'] = f"Could not convert page {target_page} to image"
        return result
    
    # Process all queries at once with Donut
    extraction_results = extract_information_from_document(image, QUERIES)
    
    # Add extraction results to the result dict
    result.update(extraction_results)
    result['processed'] = True
    
    # Save individual result
    save_individual_result(result)
    
    return result

In [94]:
# Process all documents
processed_count = 0
failed_count = 0

if not df_forms.empty:
    print(f"Processing {len(df_forms)} documents...")
    
    for idx, row in tqdm(df_forms.iterrows(), total=len(df_forms), desc="Processing documents"):
        result = process_document(row)
        
        if result['processed']:
            processed_count += 1
            # Print progress for first few successful extractions
            if processed_count <= 5:
                print(f"\nProcessed: {result['filename']} (page {result['target_page']})")
                for query in QUERIES:
                    query_key = f"{query.lower().replace(' ', '_')}_answer"
                    conf_key = f"{query.lower().replace(' ', '_')}_confidence"
                    print(f"  {query}: '{result[query_key]}' (confidence: {result[conf_key]:.3f})")
        else:
            failed_count += 1
            if 'error' in result:
                print(f"Failed to process {result['filename']}: {result['error']}")

else:
    print("No documents to process")

# Final summary
print(f"\nProcessing complete!")
print(f"Successfully processed: {processed_count}")
print(f"Failed: {failed_count}")
print(f"Individual results saved to: {RESULTS_DIR}/*.json")

Processing 10 documents...


Processing documents:   0%|                                                                    | 0/10 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Task prompt '<s_docvqa>' output: AGE'S PURCHASE IS ARE'S PURCHASE IS PURCHASE IS PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE...
Successfully parsed JSON with '<s_docvqa>': {'text_sequence': "AGE'S PURCHASE IS ARE'S PURCHASE IS PURCHASE IS PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE PURCHASE>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Task prompt '<s>' output: <s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s...
Successfully parsed JSON with '<s>': {'text_sequence': '<s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s><s>

...

Task prompt '<s_docvqa>' output: Articnt B vson of Emergency Preparedness<sep/> kindness<sep/> BOX WD.FACOG PHP Sub-awardeepsics Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Inte...
Successfully parsed JSON with '<s_docvqa>': {'text_sequence': 'Articnt B vson of Emergency Preparedness<sep/> kindness<sep/> BOX WD.FACOG PHP Sub-awardeepsics Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Interface Interfaceload Number Interface Interfaceload Number Interface Interfaceloads optimization Interface Interfaceload Number Interface Interfaceload Interfaceload optimization Interfaceload module module module module module module module module module module module module module module module module module module module module module module module module module module module module module module module m

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Task prompt '<s>' output: <s> 15 15 15 15 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 2...
Successfully parsed JSON with '<s>': {'text_sequence': '<s> 15 15 15 15 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26 26973<sep/> 18 18 18 18 18 18 18 18 18 18 18 19 18 19 18 19 18 26973189 Alba Flairston Island Stoff 18 2697389 to 19389 to 19389 to 19389 to 19389 to 19389.189 SHOTiano Road at 18 2629 of 18 2629 of 18 2629 of 18

Processing documents: 100%|███████████████████████████████████████████████████████████| 10/10 [06:00<00:00, 36.05s/it]

Task prompt '<s_cord-v2>' output: <s_menu><s_nm> Erle.J.Rolcombo</s_nm><s_discountprice> Division of Emergency</s_nm><s_price> Attachment B</s_price><sep/><s_nm> Kyselman Box WD. FACOG</s_nm><s_discountprice> PHP Sub-awarde</s_discoun...
Successfully parsed JSON with '<s_cord-v2>': {'menu': [{'nm': 'Erle.J.Rolcombo', 'discountprice': [{'price': 'Attachment B'}, {'nm': 'Kyselman Box WD. FACOG'}]}, {'nm': 'Nume of Dorganication District 9 Healthcare Coalition (EIN)', 'num': '93.817', 'price': '276973'}, {'nm': {'discountprice': '2020', 'price': '2020'}, 'price': {'num': {'nm': 'An invoice for requesting funds will be provided upon grant execution. Final invoicing is due on before April 15. 2020.'}}}]}
Nested search match for 'What is EDS Number?': [{'price': 'Attachment B'}, {'nm': 'Kyselman Box WD. FACOG'}]
Nested search match for 'What is Date Prepared?': [{'price': 'Attachment B'}, {'nm': 'Kyselman Box WD. FACOG'}]
New best results with prompt '<s_cord-v2>' (total confidence: 0.96)

P

In [ ]:
# Results are automatically saved as individual JSON files during processing
# Each file uses the same filename as the PDF but with .json extension
# 
# Example outputs:
# - Input: "0000000000000000000009508-012.pdf" 
# - Output: "0000000000000000000009508-012.json"
#
# Each JSON file contains:
# - filename: original PDF filename
# - target_page: page number that was processed
# - processed: boolean indicating success
# - {query}_answer: extracted answer for each query
# - {query}_confidence: confidence score for each answer
# - error: error message if processing failed

print(f"All results are saved individually in: {RESULTS_DIR}")
print(f"File pattern: *.json (same name as original PDFs)")

## Next Steps

1. Review the extraction results and confidence scores
2. Add more queries by modifying the `QUERIES` list above
3. Adjust the model or preprocessing if needed
4. Consider post-processing steps to clean extracted text
5. Evaluate extraction quality on a sample of documents